# 08_dani_hybrid_tuned
## Hyperparameter Tuning (Optuna) untuk Hybrid Residual LSTM — Per-Slot Search

Melanjutkan `07_dani_hybrid_residual_lstm.ipynb`. Notebook itu sudah menunjukkan **Hybrid MSTL
mengungguli Tuned Direct (nb06)** untuk CI (RMSE 2.03 vs 2.26) dan RE (RMSE 0.22 vs 0.24), dan MSTL
konsisten mengungguli Prophet di semua 6 slot. Kurva multistep juga sehat (plateau, tidak meledak).

### Scope tuning: 12 search independen

Setiap kombinasi **decomposition × slot** (`{mstl, prophet} × {Uni-LSTM CI, Uni-LSTM RE, Multi-LSTM,
BiLSTM CI, BiLSTM RE, BiLSTM Multi}`) mendapat Optuna study-nya sendiri — **75-100 trial per search**
(`N_TRIALS` di bawah, default 90, gampang diturunin kalau kelamaan).

**Yang di-search**: `look_back`, jumlah layer (stacking), `units`, `dropout`, `learning_rate`, `batch_size`.
**Yang TIDAK di-search** (tetap fixed sesuai definisi slot): tipe layer (LSTM vs BiLSTM) dan target
group (CI/RE/Multi) — karena itu identitas slot itu sendiri, bukan hyperparameter.

**Objective**: `val_loss` (MSE) dari model residual — untuk slot Uni-CI/Uni-RE/Bi-CI/Bi-RE itu MSE
1 target; untuk slot Multi itu otomatis rata-rata CI+RE (karena kedua target sudah di-MinMax-scale ke
[0,1], MSE gabungan = rata-rata setara, sama seperti konvensi nb06).

### Estimasi biaya compute

~900-1200 training run total (12 search × ~90 trial). Untuk efisiensi:
- **Fase search**: epoch dikurangi (`TRIAL_EPOCHS=25`, patience=5) + Optuna pruning (buang trial yang
  jelas tidak menjanjikan lebih awal).
- **Fase final**: hyperparameter terbaik di-retrain dengan epoch penuh (`FINAL_EPOCHS=50`, patience=10),
  konsisten dengan konvensi 04-07.

Kalau ini kelamaan di mesin lu, tinggal turunin `N_TRIALS` (misal ke 30-40) — semua kode lain otomatis
mengikuti tanpa perlu diubah.

In [ ]:
from pathlib import Path
import sys
import warnings
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

project_root = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "src").exists()),
    Path.cwd()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import TimeSeriesDataLoader

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from prophet import Prophet

try:
    import optuna
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "-q"])
    import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Project root:", project_root)
print("TensorFlow:", tf.__version__)
print("Optuna:", optuna.__version__)

In [ ]:
feature_path = (
    project_root / "data" / "processed" / "normalized" / "dataset_feature_engineered.csv"
)

df = pd.read_csv(feature_path)
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
df = df.set_index("datetime").sort_index()

CI_COL = "carbon_intensity"
RE_COL = "renewable_percentage"

df_core = df.dropna(subset=[CI_COL, RE_COL]).copy()

TRAIN_RATIO = 0.8
train_size = int(len(df_core) * TRAIN_RATIO)
train_raw = df_core.iloc[:train_size].copy()
test_raw = df_core.iloc[train_size:].copy()
TRAIN_END_INDEX = train_raw.index[-1]

print("Train:", train_raw.shape, "| Test:", test_raw.shape)

In [ ]:
TEMPORAL_CYCLICAL = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos"]

loader_time = TimeSeriesDataLoader()
all_index_df = pd.DataFrame(index=df_core.index)
cyclical_full = loader_time.add_time_features(all_index_df)[TEMPORAL_CYCLICAL].copy()

print("Cyclical features ready.")

## 1. Decomposition (reused exactly from nb07 — causal MSTL + leak-free Prophet)

In [ ]:
def compute_causal_decomposition(series, train_end_index, trend_window=168):
    trend = series.rolling(window=trend_window, min_periods=1).mean()
    deseasonalized = series - trend

    train_deseasonalized = deseasonalized.loc[:train_end_index]

    hour_profile = train_deseasonalized.groupby(train_deseasonalized.index.hour).mean()
    seasonal_24 = pd.Series(series.index.hour.map(hour_profile).astype(float), index=series.index)

    residual_after_24 = (
        train_deseasonalized.values
        - train_deseasonalized.index.hour.map(hour_profile).astype(float)
    )
    weekhour_key_train = train_deseasonalized.index.weekday * 24 + train_deseasonalized.index.hour
    weekhour_profile = pd.Series(residual_after_24, index=weekhour_key_train).groupby(level=0).mean()
    weekhour_key_full = series.index.weekday * 24 + series.index.hour
    seasonal_168 = pd.Series(weekhour_key_full.map(weekhour_profile).astype(float), index=series.index)

    resid = series - trend - seasonal_24 - seasonal_168

    return pd.DataFrame(
        {"trend": trend, "seasonal_24": seasonal_24, "seasonal_168": seasonal_168, "resid": resid},
        index=series.index
    )


mstl_ci = compute_causal_decomposition(df_core[CI_COL], TRAIN_END_INDEX, trend_window=168)
mstl_re = compute_causal_decomposition(df_core[RE_COL], TRAIN_END_INDEX, trend_window=168)

mstl_decomposition = {
    "CI_reconstruction": mstl_ci["trend"] + mstl_ci["seasonal_24"] + mstl_ci["seasonal_168"],
    "CI_resid": mstl_ci["resid"],
    "RE_reconstruction": mstl_re["trend"] + mstl_re["seasonal_24"] + mstl_re["seasonal_168"],
    "RE_resid": mstl_re["resid"],
}
print("MSTL decomposition ready.")

In [ ]:
def run_prophet_decomposition(target_col):
    prophet_df = df_core.reset_index()[["datetime", target_col]].copy()
    prophet_df["datetime"] = prophet_df["datetime"].dt.tz_localize(None)
    prophet_df.columns = ["ds", "y"]

    train_prophet_df = prophet_df.iloc[:train_size]
    test_prophet_df = prophet_df.iloc[train_size:]

    model = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=True)
    model.fit(train_prophet_df)

    future = model.make_future_dataframe(periods=len(test_prophet_df), freq="h")
    forecast = model.predict(future)

    assert len(forecast) == len(df_core), "Prophet future length mismatch - check for gaps."

    reconstruction = pd.Series(forecast["yhat"].values, index=df_core.index)
    resid = df_core[target_col] - reconstruction
    return reconstruction, resid


print("Fitting Prophet on CI...")
ci_prophet_reconstruction, ci_prophet_resid = run_prophet_decomposition(CI_COL)
print("Fitting Prophet on RE...")
re_prophet_reconstruction, re_prophet_resid = run_prophet_decomposition(RE_COL)

prophet_decomposition = {
    "CI_reconstruction": ci_prophet_reconstruction,
    "CI_resid": ci_prophet_resid,
    "RE_reconstruction": re_prophet_reconstruction,
    "RE_resid": re_prophet_resid,
}

decompositions = {"mstl": mstl_decomposition, "prophet": prophet_decomposition}
print("Prophet decomposition ready.")

## 2. Residual dataset helpers (reused from nb07)

In [ ]:
def build_residual_dataframe(decomp):
    df_resid = pd.DataFrame(index=df_core.index)
    df_resid["CI_resid"] = decomp["CI_resid"]
    df_resid["RE_resid"] = decomp["RE_resid"]
    df_resid[TEMPORAL_CYCLICAL] = cyclical_full[TEMPORAL_CYCLICAL]
    return df_resid


def get_feature_target_cols(target_group):
    if target_group == "CI":
        return ["CI_resid", *TEMPORAL_CYCLICAL], ["CI_resid"]
    elif target_group == "RE":
        return ["RE_resid", *TEMPORAL_CYCLICAL], ["RE_resid"]
    elif target_group == "MULTI":
        return ["CI_resid", "RE_resid", *TEMPORAL_CYCLICAL], ["CI_resid", "RE_resid"]
    raise ValueError("target_group must be CI, RE, or MULTI")


def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    nonzero = y_true != 0
    mape = (
        np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
        if np.any(nonzero) else np.nan
    )
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot

    return {"MAE": mae, "RMSE": rmse, "MAPE (%)": mape, "R2": r2}

## 3. Model builder (shared core between search-time and final retrain) + Optuna pruning callback

In [ ]:
class OptunaPruningCallback(tf.keras.callbacks.Callback):
    def __init__(self, trial, monitor="val_loss"):
        super().__init__()
        self.trial = trial
        self.monitor = monitor

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current = logs.get(self.monitor)
        if current is None:
            return
        self.trial.report(current, step=epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()


def build_model_core(architecture, input_shape, output_dim, n_layers, units, dropout_rate, learning_rate):
    model = Sequential()
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        if architecture == "lstm":
            if i == 0:
                model.add(LSTM(units, return_sequences=return_sequences, input_shape=input_shape))
            else:
                model.add(LSTM(units, return_sequences=return_sequences))
        else:
            if i == 0:
                model.add(Bidirectional(LSTM(units, return_sequences=return_sequences), input_shape=input_shape))
            else:
                model.add(Bidirectional(LSTM(units, return_sequences=return_sequences)))
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))

    model.add(Dense(output_dim))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=["mae"]
    )
    return model


def build_model_for_trial(trial, architecture, input_shape, output_dim):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    units = trial.suggest_categorical("units", [32, 64, 96, 128])
    dropout_rate = trial.suggest_float("dropout", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    return build_model_core(architecture, input_shape, output_dim, n_layers, units, dropout_rate, learning_rate)


def build_model_from_params(params, architecture, input_shape, output_dim):
    return build_model_core(
        architecture, input_shape, output_dim,
        params["n_layers"], params["units"], params["dropout"], params["learning_rate"]
    )

## 4. Configuration

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

FORECAST_HORIZON = 1
VALIDATION_SPLIT = 0.1

N_TRIALS = 90          # per search; turunin ke 30-40 kalau kelamaan
TRIAL_EPOCHS = 25
TRIAL_PATIENCE = 5

FINAL_EPOCHS = 50
FINAL_PATIENCE = 10

LOOK_BACK_CANDIDATES = [6, 12, 24, 48]
BATCH_SIZE_CANDIDATES = [16, 32, 64]

MULTISTEP_HORIZON = 24
ORIGIN_STRIDE = 24
N_ORIGINS_MAX = 100

experiment_specs = [
    ("Uni-LSTM CI", "CI", "lstm"),
    ("Uni-LSTM RE", "RE", "lstm"),
    ("Multi-LSTM", "MULTI", "lstm"),
    ("BiLSTM CI", "CI", "bilstm"),
    ("BiLSTM RE", "RE", "bilstm"),
    ("BiLSTM Multi", "MULTI", "bilstm"),
]
decomposition_methods = ["mstl", "prophet"]

print("Total independent searches:", len(experiment_specs) * len(decomposition_methods))
print("Estimated total trials:", len(experiment_specs) * len(decomposition_methods) * N_TRIALS)

## 5. Objective function generator

In [ ]:
def make_objective(df_resid, feature_cols, target_cols, architecture):
    def objective(trial):
        look_back = trial.suggest_categorical("look_back", LOOK_BACK_CANDIDATES)
        batch_size = trial.suggest_categorical("batch_size", BATCH_SIZE_CANDIDATES)

        loader = TimeSeriesDataLoader()
        train_scaled, _ = loader.split_and_scale(
            df=df_resid, feature_cols=feature_cols, target_cols=target_cols, train_ratio=TRAIN_RATIO
        )
        X_train, y_train = loader.create_sliding_window(
            train_scaled, target_cols=target_cols, look_back=look_back, forecast_horizon=FORECAST_HORIZON
        )

        model = build_model_for_trial(
            trial, architecture, input_shape=(X_train.shape[1], X_train.shape[2]), output_dim=len(target_cols)
        )

        pruning_cb = OptunaPruningCallback(trial, monitor="val_loss")
        early_stop = EarlyStopping(monitor="val_loss", patience=TRIAL_PATIENCE, restore_best_weights=True)

        history = model.fit(
            X_train, y_train,
            validation_split=VALIDATION_SPLIT,
            epochs=TRIAL_EPOCHS,
            batch_size=batch_size,
            callbacks=[early_stop, pruning_cb],
            shuffle=False,
            verbose=0
        )

        return min(history.history["val_loss"])

    return objective

## 6. Rolling-origin recursive multistep forecast (reused from nb07, `look_back` now per-config)

In [ ]:
def rolling_origin_recursive_forecast(
    model, loader, test_scaled, feature_cols, target_cols,
    horizon, look_back, stride=ORIGIN_STRIDE, n_origins_max=N_ORIGINS_MAX
):
    data = test_scaled[feature_cols].values
    n = len(data)

    origins = list(range(look_back, n - horizon, stride))[:n_origins_max]
    target_idx = [feature_cols.index(c) for c in target_cols]

    per_origin_preds = []

    for origin in origins:
        history_window = data[origin - look_back: origin].copy()
        preds_scaled = []

        for h in range(horizon):
            X_step = history_window[np.newaxis, :, :]
            pred_scaled = model.predict(X_step, verbose=0).reshape(-1)
            preds_scaled.append(pred_scaled)

            next_row = data[origin + h].copy()
            for i, idx in enumerate(target_idx):
                next_row[idx] = pred_scaled[i]

            history_window = np.vstack([history_window[1:], next_row])

        preds_scaled = np.array(preds_scaled)
        preds_inv = loader.inverse_transform_predictions(
            preds_scaled, target_cols=target_cols, feature_cols=feature_cols
        )
        per_origin_preds.append((origin, preds_inv))

    return origins, per_origin_preds

## 7. Main loop — 12 independent search + final retrain + evaluation

Ini bagian yang paling lama jalannya. Progress di-print per search (durasi, best val_loss, best params)
supaya bisa dipantau.

In [ ]:
all_studies = {}
all_best_params = []
all_results = []
all_horizon_results = []
all_models = {}
all_predictions = {}

overall_start = time.time()

for method in decomposition_methods:
    decomp = decompositions[method]

    for model_name, target_group, architecture in experiment_specs:
        print("\n" + "=" * 80)
        print(f"[{time.time() - overall_start:7.1f}s elapsed] SEARCH: {model_name} | {method}")
        print("=" * 80)

        feature_cols, target_cols = get_feature_target_cols(target_group)
        df_resid = build_residual_dataframe(decomp)

        objective = make_objective(df_resid, feature_cols, target_cols, architecture)

        sampler = optuna.samplers.TPESampler(seed=SEED)
        pruner = optuna.pruners.MedianPruner(n_warmup_steps=5)
        study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)

        t0 = time.time()
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
        search_elapsed = time.time() - t0

        print(f"Search done in {search_elapsed/60:.1f} min | best val_loss={study.best_value:.6f}")
        print("Best params:", study.best_params)

        all_studies[(method, model_name)] = study
        all_best_params.append({
            "Decomposition": method, "Model": model_name,
            **study.best_params,
            "best_val_loss": study.best_value,
            "search_time_min": search_elapsed / 60
        })

        # --- final retrain with best params, full epoch budget ---
        best = study.best_params
        look_back = best["look_back"]
        batch_size = best["batch_size"]

        loader = TimeSeriesDataLoader()
        train_scaled, test_scaled = loader.split_and_scale(
            df=df_resid, feature_cols=feature_cols, target_cols=target_cols, train_ratio=TRAIN_RATIO
        )
        X_train, y_train = loader.create_sliding_window(
            train_scaled, target_cols=target_cols, look_back=look_back, forecast_horizon=FORECAST_HORIZON
        )
        X_test, y_test = loader.create_sliding_window(
            test_scaled, target_cols=target_cols, look_back=look_back, forecast_horizon=FORECAST_HORIZON
        )

        final_model = build_model_from_params(
            best, architecture, input_shape=(X_train.shape[1], X_train.shape[2]), output_dim=len(target_cols)
        )

        final_model.fit(
            X_train, y_train,
            validation_split=VALIDATION_SPLIT,
            epochs=FINAL_EPOCHS,
            batch_size=batch_size,
            callbacks=[EarlyStopping(monitor="val_loss", patience=FINAL_PATIENCE, restore_best_weights=True)],
            shuffle=False,
            verbose=0
        )

        # --- one-step evaluation ---
        pred_scaled = final_model.predict(X_test, verbose=0)
        if len(target_cols) == 1:
            pred_scaled = pred_scaled.reshape(-1)

        resid_pred_inv = loader.inverse_transform_predictions(
            pred_scaled, target_cols=target_cols, feature_cols=feature_cols
        )
        plot_test_index = test_raw.index[look_back:]

        if target_group == "CI":
            final_pred = resid_pred_inv.reshape(-1) + decomp["CI_reconstruction"].loc[plot_test_index].values
            actual = df_core.loc[plot_test_index, CI_COL].values
            metrics = evaluate(actual, final_pred)
            all_results.append({"Decomposition": method, "Model": model_name, "Target": "CI", **metrics})
            all_predictions[(method, model_name)] = {"CI": final_pred, "RE": None}

        elif target_group == "RE":
            final_pred = resid_pred_inv.reshape(-1) + decomp["RE_reconstruction"].loc[plot_test_index].values
            actual = df_core.loc[plot_test_index, RE_COL].values
            metrics = evaluate(actual, final_pred)
            all_results.append({"Decomposition": method, "Model": model_name, "Target": "RE", **metrics})
            all_predictions[(method, model_name)] = {"CI": None, "RE": final_pred}

        else:
            ci_final = resid_pred_inv[:, 0] + decomp["CI_reconstruction"].loc[plot_test_index].values
            re_final = resid_pred_inv[:, 1] + decomp["RE_reconstruction"].loc[plot_test_index].values
            ci_metrics = evaluate(df_core.loc[plot_test_index, CI_COL].values, ci_final)
            re_metrics = evaluate(df_core.loc[plot_test_index, RE_COL].values, re_final)
            all_results.append({"Decomposition": method, "Model": model_name, "Target": "CI", **ci_metrics})
            all_results.append({"Decomposition": method, "Model": model_name, "Target": "RE", **re_metrics})
            all_predictions[(method, model_name)] = {"CI": ci_final, "RE": re_final}

        # --- rolling-origin recursive multistep evaluation ---
        origins, per_origin_preds = rolling_origin_recursive_forecast(
            final_model, loader, test_scaled, feature_cols, target_cols,
            horizon=MULTISTEP_HORIZON, look_back=look_back
        )

        for origin, preds_inv in per_origin_preds:
            preds_inv = np.atleast_2d(preds_inv)
            if preds_inv.shape[0] == 1 and len(target_cols) > 1:
                preds_inv = preds_inv.reshape(-1, len(target_cols))
            elif preds_inv.ndim == 1:
                preds_inv = preds_inv.reshape(-1, 1)

            origin_index = test_scaled.index[origin: origin + MULTISTEP_HORIZON]

            for h in range(MULTISTEP_HORIZON):
                ts = origin_index[h]
                for i, col in enumerate(target_cols):
                    target_name = "CI" if col.startswith("CI") else "RE"
                    recon_val = decomp[f"{target_name}_reconstruction"].loc[ts]
                    pred_final = preds_inv[h, i] + recon_val
                    actual_final = df_core.loc[ts, CI_COL if target_name == "CI" else RE_COL]

                    all_horizon_results.append({
                        "Decomposition": method, "Model": model_name, "Target": target_name,
                        "Horizon": h + 1, "AbsError": abs(actual_final - pred_final)
                    })

        all_models[(method, model_name)] = {
            "model": final_model, "feature_cols": feature_cols, "target_cols": target_cols,
            "look_back": look_back, "best_params": best
        }

        print(f"Completed: {model_name} | {method} [total elapsed: {(time.time()-overall_start)/60:.1f} min]")

print(f"\n\nALL SEARCHES DONE. Total time: {(time.time()-overall_start)/60:.1f} min")

In [ ]:
best_params_df = pd.DataFrame(all_best_params)
print("=== BEST HYPERPARAMETERS PER CONFIG ===")
display(best_params_df.sort_values(["Decomposition", "Model"]).reset_index(drop=True))

In [ ]:
results_df = pd.DataFrame(all_results)
print("=== ONE-STEP TUNED HYBRID EVALUATION ===")
display(results_df.sort_values(["Target", "Decomposition", "RMSE"]).reset_index(drop=True))

In [ ]:
# Compare against nb07 (default hyperparameters) if that CSV is available.
nb07_path = project_root / "results" / "hybrid" / "07_dani_hybrid_onestep_metrics.csv"

if nb07_path.exists():
    nb07_results = pd.read_csv(nb07_path)

    comparison = (
        results_df[["Decomposition", "Model", "Target", "RMSE"]]
        .rename(columns={"RMSE": "RMSE_tuned"})
        .merge(
            nb07_results[["Decomposition", "Model", "Target", "RMSE"]].rename(columns={"RMSE": "RMSE_nb07"}),
            on=["Decomposition", "Model", "Target"]
        )
    )
    comparison["Improvement (%)"] = (
        (comparison["RMSE_nb07"] - comparison["RMSE_tuned"]) / comparison["RMSE_nb07"] * 100
    )

    print("=== TUNED (nb08) vs DEFAULT HYPERPARAMS (nb07) ===")
    display(comparison.sort_values(["Target", "Improvement (%)"], ascending=[True, False]))
else:
    print("nb07 results CSV not found at:", nb07_path, "- skipping comparison.")

In [ ]:
horizon_results_df = pd.DataFrame(all_horizon_results)

horizon_summary = (
    horizon_results_df
    .groupby(["Decomposition", "Model", "Target", "Horizon"], as_index=False)["AbsError"]
    .mean()
    .rename(columns={"AbsError": "MAE_at_horizon"})
)

print("Rows per decomposition method:")
print(horizon_summary["Decomposition"].value_counts())

In [ ]:
for target in ["CI", "RE"]:
    fig, ax = plt.subplots(figsize=(12, 6))

    for method in decomposition_methods:
        for model_name, target_group, _ in experiment_specs:
            if target_group != "MULTI" and target_group != target:
                continue

            subset = horizon_summary[
                (horizon_summary["Decomposition"] == method)
                & (horizon_summary["Model"] == model_name)
                & (horizon_summary["Target"] == target)
            ].sort_values("Horizon")

            if len(subset) == 0:
                continue

            ax.plot(subset["Horizon"], subset["MAE_at_horizon"], label=f"{method} - {model_name}", alpha=0.8)

    ax.set_xlabel("Horizon (jam ke depan)")
    ax.set_ylabel("MAE")
    ax.set_title(f"{target} — Tuned Hybrid, MAE vs horizon (rolling-origin)")
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
results_dir = project_root / "results" / "hybrid_tuned"
results_dir.mkdir(parents=True, exist_ok=True)

best_params_df.to_csv(results_dir / "08_dani_hybrid_tuned_best_params.csv", index=False)
results_df.to_csv(results_dir / "08_dani_hybrid_tuned_onestep_metrics.csv", index=False)
horizon_summary.to_csv(results_dir / "08_dani_hybrid_tuned_multistep_horizon_mae.csv", index=False)

models_dir = project_root / "models" / "hybrid_tuned"
models_dir.mkdir(parents=True, exist_ok=True)

for (method, model_name), bundle in all_models.items():
    safe_name = model_name.lower().replace(" ", "_").replace("-", "")
    bundle["model"].save(models_dir / f"08_hybrid_tuned_{method}_{safe_name}.keras")

print("Saved results to:", results_dir)
print("Saved 12 tuned models to:", models_dir)

## Ringkasan

- **12 search independen** (`best_params_df`) — arsitektur optimal berbeda per slot & per decomposition
  method, tidak dipaksa satu ukuran untuk semua.
- **`results_df`** — evaluasi one-step, langsung dibandingkan ke `07_dani_hybrid_onestep_metrics.csv`
  (default hyperparameter) lewat tabel `comparison`.
- **`horizon_summary`** — evaluasi multistep rolling-origin, per-config, dengan `look_back` yang sudah
  disesuaikan hasil tuning masing-masing.

Yang perlu diperiksa:
1. Apakah tuning per-slot memberi improvement yang berarti dibanding nb07 (kolom `Improvement (%)`
   di tabel `comparison`) — atau overhead compute besar ini tidak sepadan dengan gain yang didapat.
2. Apakah kurva MAE-vs-horizon tetap plateau (sehat) di versi tuned ini.
3. Konfigurasi mana (MSTL vs Prophet, per slot) yang jadi kandidat final untuk dilaporkan di paper.